In [7]:
import pandas as pd
import os
import math
from datetime import datetime

# 1. Загрузка данных
file_path = "C:/Users/majkl/github_repo/WorkSpace/preprocessing_excel_data/data/Выгурзка_25.09.25_без_дубликатов.xlsx"
df = pd.read_excel(file_path)

# 2. Группировка по дате и подсчёт уникальных ID продавцов
grouped = df.groupby("Дата")["ID продавца, на которого подали жалобу"].nunique().reset_index()
grouped.columns = ["Дата", "Unique_IDs"]

# 3. Разделение дат на "большие" (>=600 ID) и "малые" (<600)
big_dates = grouped[grouped["Unique_IDs"] >= 600]["Дата"].tolist()
small_dates = grouped[grouped["Unique_IDs"] < 600].copy()

# 4. Группировка "малых" дат (лимит = 600 ID)
merged_groups = []
current_group = []
current_count = 0

for _, row in small_dates.iterrows():
    date = row["Дата"]
    unique_ids = row["Unique_IDs"]
    
    if current_count + unique_ids <= 600:
        current_group.append(date)
        current_count += unique_ids
    else:
        if current_group:
            merged_groups.append((current_group, current_count))
        current_group = [date]
        current_count = unique_ids

if current_group:
    merged_groups.append((current_group, current_count))

# 5. Создание корневой папки
root_folder = "Внешние заявители"
os.makedirs(root_folder, exist_ok=True)

# 6. Функция для записи CSV с уникальными SKU по группам продавцов
def save_sku_parts(data, folder_path, group_name):
    """Сохраняет уникальные SKU для каждой группы продавцов."""
    unique_sellers = data["ID продавца, на которого подали жалобу"].unique()
    num_parts = math.ceil(len(unique_sellers) / 600)
    
    for i in range(num_parts):
        start_idx = i * 600
        end_idx = start_idx + 600
        sellers_part = unique_sellers[start_idx:end_idx]
        
        # Фильтруем SKU для текущей группы продавцов
        part_data = data[data["ID продавца, на которого подали жалобу"].isin(sellers_part)]
        unique_skus = part_data["SKU"].dropna().unique()  # Убираем NaN
        
        # Сохраняем только уникальные SKU в CSV
        pd.DataFrame(unique_skus, columns=["SKU"]).to_csv(
            os.path.join(folder_path, f"sku_part_{i+1}.csv"), 
            index=False
        )

# 7. Сохранение данных и отчет
group_dataframes = []

def save_group_data(dates, group_name, total_ids):
    """Сохраняет данные группы и CSV с уникальными SKU."""
    group_folder = os.path.join(root_folder, group_name)
    os.makedirs(group_folder, exist_ok=True)
    
    # Фильтрация данных по группе дат
    mask = df["Дата"].isin(dates)
    group_df = df[mask]
    
    # Сохранение Excel с полными данными
    group_df.to_excel(os.path.join(group_folder, f"data_{group_name}.xlsx"), index=False)
    
    # Сохраняем уникальные SKU по группам продавцов
    save_sku_parts(group_df, group_folder, group_name)
    
    # Добавление в отчет
    for date in dates:
        group_dataframes.append({
            "Дата": date,
            "Уникальные ID продавцов": grouped[grouped["Дата"] == date]["Unique_IDs"].iloc[0],
            "Группа дат": group_name,
            "Уникальные ID в группе": total_ids
        })

# 8. Сохранение "больших" дат (отдельно)
for date in big_dates:
    group_name = date.strftime("%Y-%m-%d")
    total_ids = grouped[grouped["Дата"] == date]["Unique_IDs"].iloc[0]
    save_group_data([date], group_name, total_ids)

# 9. Сохранение сгруппированных "малых" дат
for i, (group_dates, group_total) in enumerate(merged_groups, 1):
    date_min = min(group_dates).strftime("%Y-%m-%d")
    date_max = max(group_dates).strftime("%Y-%m-%d")
    group_name = f"{date_min}_to_{date_max}"
    save_group_data(group_dates, group_name, group_total)

# 10. Сохранение отчета
report_df = pd.DataFrame(group_dataframes)
report_df.to_excel(os.path.join(root_folder, "report.xlsx"), index=False)

print(f"Готово! Результаты сохранены в папке '{root_folder}'.")

Готово! Результаты сохранены в папке 'Внешние заявители'.


In [1]:
import pandas as pd
import os
import math
from datetime import datetime

# Конфигурация
INPUT_FILE = "C:/Users/majkl/github_repo/WorkSpace/preprocessing_excel_data/data/Выгурзка_25.09.25_без_дубликатов.xlsx"
OUTPUT_FOLDER = "Внешние_заявители_группировка"  # Измененное название папки

def group_dates_by_sellers(df):
    """Группирует даты по количеству уникальных ID продавцов"""
    grouped = df.groupby("Дата")["ID продавца, на которого подали жалобу"].nunique().reset_index()
    grouped.columns = ["Дата", "Unique_IDs"]
    
    # Разделение дат
    big_dates = grouped[grouped["Unique_IDs"] >= 600]["Дата"].tolist()
    small_dates = grouped[grouped["Unique_IDs"] < 600].copy()
    
    # Группировка малых дат
    merged_groups = []
    current_group = []
    current_count = 0
    
    for _, row in small_dates.iterrows():
        date = row["Дата"]
        unique_ids = row["Unique_IDs"]
        
        if current_count + unique_ids <= 600:
            current_group.append(date)
            current_count += unique_ids
        else:
            if current_group:
                merged_groups.append((current_group, current_count))
            current_group = [date]
            current_count = unique_ids
    
    if current_group:
        merged_groups.append((current_group, current_count))
    
    return big_dates, merged_groups

def save_sku_parts(data, folder_path):
    """Сохраняет уникальные SKU по группам продавцов"""
    unique_sellers = data["ID продавца, на которого подали жалобу"].unique()
    num_parts = math.ceil(len(unique_sellers) / 600)
    
    for i in range(num_parts):
        start_idx = i * 600
        end_idx = start_idx + 600
        sellers_part = unique_sellers[start_idx:end_idx]
        part_data = data[data["ID продавца, на которого подали жалобу"].isin(sellers_part)]
        unique_skus = part_data["SKU"].dropna().unique()
        
        pd.DataFrame(unique_skus, columns=["SKU"]).to_csv(
            os.path.join(folder_path, f"sku_part_{i+1}.csv"), 
            index=False
        )

def process_data(df, output_folder):
    """Основная функция обработки данных"""
    os.makedirs(output_folder, exist_ok=True)
    big_dates, merged_groups = group_dates_by_sellers(df)
    report_data = []

    # Обработка больших дат
    for date in big_dates:
        group_name = date.strftime("%Y-%m-%d")
        group_folder = os.path.join(output_folder, group_name)
        os.makedirs(group_folder, exist_ok=True)
        
        mask = df["Дата"] == date
        group_df = df[mask]
        
        group_df.to_excel(os.path.join(group_folder, f"data_{group_name}.xlsx"), index=False)
        save_sku_parts(group_df, group_folder)
        
        report_data.append({
            "Дата": date,
            "Уникальные ID продавцов": len(group_df["ID продавца, на которого подали жалобу"].unique()),
            "Группа дат": group_name,
            "Уникальные ID в группе": len(group_df["ID продавца, на которого подали жалобу"].unique())
        })

    # Обработка малых дат
    for i, (group_dates, group_total) in enumerate(merged_groups, 1):
        date_min = min(group_dates).strftime("%Y-%m-%d")
        date_max = max(group_dates).strftime("%Y-%m-%d")
        group_name = f"{date_min}_to_{date_max}"
        group_folder = os.path.join(output_folder, group_name)
        os.makedirs(group_folder, exist_ok=True)
        
        mask = df["Дата"].isin(group_dates)
        group_df = df[mask]
        
        group_df.to_excel(os.path.join(group_folder, f"data_{group_name}.xlsx"), index=False)
        save_sku_parts(group_df, group_folder)
        
        for date in group_dates:
            report_data.append({
                "Дата": date,
                "Уникальные ID продавцов": len(df[df["Дата"] == date]["ID продавца, на которого подали жалобу"].unique()),
                "Группа дат": group_name,
                "Уникальные ID в группе": group_total
            })

    # Сохранение отчета
    pd.DataFrame(report_data).to_excel(os.path.join(output_folder, "report.xlsx"), index=False)


    df = pd.read_excel(INPUT_FILE)
    process_data(df, OUTPUT_FOLDER)
    print(f"Готово! Результаты сохранены в папке '{OUTPUT_FOLDER}'")